In [108]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

# **Disease Classification**

## 1. Load Data

In [109]:
df = pd.read_csv("symtoms_df.csv")
df.head()

,Unnamed: 0,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4
0,0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches
1,1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN
2,2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN
3,3,Fungal infection,itching,skin_rash,dischromic _patches,NaN
4,4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN


In [110]:
df.isnull().sum()

Unnamed: 0      0
Disease         0
Symptom_1       0
Symptom_2       0
Symptom_3       0
Symptom_4     348
dtype: int64

In [111]:
df.duplicated().sum()

np.int64(0)

In [112]:
df = df.drop(columns = ["Unnamed: 0"])

In [113]:
df["Disease"] = df["Disease"].str.strip()

In [114]:
df.shape

(4920, 5)

## 2. Feature Engineering

In [115]:
symptom_cols = [
    col for col in df.columns
    if col.lower().startswith("symptom_")
]

In [116]:
print("Symptom columns:", symptom_cols)
print("Number of symptom columns:", len(symptom_cols))

Symptom columns: ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']
Number of symptom columns: 4


In [117]:
df["Disease"].unique()

array(['Fungal infection', 'Allergy', 'GERD', 'Chronic cholestasis',
       'Drug Reaction', 'Peptic ulcer diseae', 'AIDS', 'Diabetes',
       'Gastroenteritis', 'Bronchial Asthma', 'Hypertension', 'Migraine',
       'Cervical spondylosis', 'Paralysis (brain hemorrhage)', 'Jaundice',
       'Malaria', 'Chicken pox', 'Dengue', 'Typhoid', 'hepatitis A',
       'Hepatitis B', 'Hepatitis C', 'Hepatitis D', 'Hepatitis E',
       'Alcoholic hepatitis', 'Tuberculosis', 'Common Cold', 'Pneumonia',
       'Dimorphic hemmorhoids(piles)', 'Heart attack', 'Varicose veins',
       'Hypothyroidism', 'Hyperthyroidism', 'Hypoglycemia',
       'Osteoarthristis', 'Arthritis',
       '(vertigo) Paroymsal  Positional Vertigo', 'Acne',
       'Urinary tract infection', 'Psoriasis', 'Impetigo'], dtype=object)

## 3. Text Preprocessing

In [118]:
def normalize_symptom(value):

    if pd.isna(value):
        return ""

    return re.sub(r"[\s_]+","_",str(value).strip().lower()).strip("_")

In [ ]:
symptom_cols = [c for c in df.columns if c.startswith("Symptom_")]
for col in symptom_cols:
    df[col] = df[col].fillna("").str.strip().str.lower()
    df[col] = df[col].str.replace(" ", "", regex=False)

In [ ]:
df["symptom_list"] = df[symptom_cols].apply(lambda row: sorted({normalize_symptom(v) for v in row} - {""}), axis=1)

In [ ]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,symptom_list
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic_patches,"[dischromic_patches, itching, nodal_skin_erupt..."
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic_patches,,"[dischromic_patches, nodal_skin_eruptions, ski..."
2,Fungal infection,itching,nodal_skin_eruptions,dischromic_patches,,"[dischromic_patches, itching, nodal_skin_erupt..."
3,Fungal infection,itching,skin_rash,dischromic_patches,,"[dischromic_patches, itching, skin_rash]"
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,,"[itching, nodal_skin_eruptions, skin_rash]"


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df["symptom_list"])
y = df["Disease"]
print("Feature matrix:", X.shape, "| unique symptoms:", len(mlb.classes_))

Feature matrix: (4920, 86) | unique symptoms: 86


In [ ]:
mlb

MultiLabelBinarizer()

In [ ]:
mlb.classes_

array(['abdominal_pain', 'acidity', 'altered_sensorium', 'anxiety',
       'back_pain', 'blackheads', 'bladder_discomfort', 'blister',
       'bloody_stool', 'blurred_and_distorted_vision', 'breathlessness',
       'bruising', 'burning_micturition', 'chest_pain', 'chills',
       'cold_hands_and_feets', 'constipation', 'continuous_feel_of_urine',
       'continuous_sneezing', 'cough', 'cramps', 'dark_urine',
       'dehydration', 'diarrhoea', 'dischromic_patches',
       'distention_of_abdomen', 'dizziness', 'excessive_hunger',
       'extra_marital_contacts', 'family_history', 'fatigue',
       'foul_smell_of_urine', 'headache', 'high_fever', 'hip_joint_pain',
       'indigestion', 'irregular_sugar_level', 'irritation_in_anus',
       'itching', 'joint_pain', 'knee_pain', 'lack_of_concentration',
       'lethargy', 'loss_of_appetite', 'loss_of_balance', 'mood_swings',
       'movement_stiffness', 'muscle_wasting', 'muscle_weakness',
       'nausea', 'neck_pain', 'nodal_skin_eruptions'

## 4. Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

##5. Model Creation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
model = LogisticRegression()
model.fit(X_train,y_train)

/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn

LogisticRegression()

## 6. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.9969512195121951

Classification Report:
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        24
                                   AIDS       1.00      1.00      1.00        24
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        24
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        24
                       Bronchial Asthma       0.96      1.00      0.98        24
                   Cervical spondylosis       1.00      1.00      1.00        24
                            Chicken pox       1.00      1.00      1.00        24
                    Chronic cholestasis       1.00      1.00      1.00        24
                            Common Cold       1.00    

/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [ ]:
print("Train Accuracy:",model.score(X_train, y_train))
print("Test Accuracy:",model.score(X_test, y_test))

Train Accuracy: 0.9946646341463414
Test Accuracy: 0.9969512195121951


/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 7. Prediction

In [ ]:
def predict(symptoms, top_k=3):
    known = [s for s in symptoms if s in mlb.classes_]
    probs = model.predict_proba(mlb.transform([known]))[0]
    top = probs.argsort()[::-1][:top_k]
    return [(model.classes_[i], round(probs[i], 3)) for i in top]

disease = predict(["abdominal_pain", "acidity", "petdard"])[0][0]
print(disease)

Migraine


/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nightshift5/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 8. Save Model

In [ ]:
import joblib

In [ ]:
joblib.dump({"model": model , "mlb": mlb}, "disease_model.joblib")

['disease_model.joblib']



---



# **Precaution Recommendations**




In [ ]:
df2 = pd.read_csv("precautions_df.csv")

In [ ]:
df2.head()

,Unnamed: 0,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [ ]:
df2.isnull().sum()

Unnamed: 0      0
Disease         0
Precaution_1    0
Precaution_2    0
Precaution_3    1
Precaution_4    1
dtype: int64

In [ ]:
df2.duplicated().sum()

np.int64(0)

In [ ]:
df2 = df2.drop(columns = ["Unnamed: 0"])

In [ ]:
df2.head()

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [ ]:
df2 = df2.fillna("")

In [ ]:
def precaution(disease) :
  precautions = df2.loc[df2['Disease'] == disease ]
  precaution_list = [p for p in precautions.values[0].tolist()[1:] if p.strip()]

  return precaution_list

In [ ]:
precaution(disease)

['meditation',
 'reduce stress',
 'use poloroid glasses in sun',
 'consult doctor']

----


# **Description**

In [ ]:
df3 = pd.read_csv("description.csv")
df3.head()

,Disease,Description
0,Fungal infection,Fungal infection is a common skin condition ca...
1,Allergy,Allergy is an immune system reaction to a subs...
2,GERD,GERD (Gastroesophageal Reflux Disease) is a di...
3,Chronic cholestasis,Chronic cholestasis is a condition where bile ...
4,Drug Reaction,Drug Reaction occurs when the body reacts adve...


In [ ]:
df3.isnull().sum()

Disease        0
Description    0
dtype: int64

In [ ]:
def description(disease):
  match = df3.loc[df3['Disease'] == disease]

  if not match.empty:
      description = match['Description'].iloc[0]

  else:
      description = "No description available for this disease."

  return description

In [ ]:
description(disease)

'Migraine is a type of headache that often involves severe pain and sensitivity to light and sound.'

---

# **Medication**

In [ ]:
df4 = pd.read_csv("medications.csv")
df4.head()

,Disease,Medication
0,Fungal infection,"['Antifungal Cream', 'Fluconazole', 'Terbinafi..."
1,Allergy,"['Antihistamines', 'Decongestants', 'Epinephri..."
2,GERD,"['Proton Pump Inhibitors (PPIs)', 'H2 Blockers..."
3,Chronic cholestasis,"['Ursodeoxycholic acid', 'Cholestyramine', 'Me..."
4,Drug Reaction,"['Antihistamines', 'Epinephrine', 'Corticoster..."


In [ ]:
def medication(disease):
  match = df4.loc[df4['Disease'] == disease]

  if not match.empty:
      import ast
      medication_list = ast.literal_eval(match['Medication'].iloc[0])
      return medication_list

  return f"No medications found for '{disease}'. Check disease name spelling in medications.csv."

In [ ]:
medication(disease)

['Analgesics',
 'Triptans',
 'Ergotamine derivatives',
 'Preventive medications',
 'Biofeedback']